In [ ]:
# Transformers undet the hood: Attention is all you need (https://arxiv.org/pdf/1706.03762)

# GPT: Generativaly Pre-trained Transformers

In [ ]:
# Dataset: Tiny Shakesphere

In [ ]:
!pwd
!ls

/content
sample_data


In [1]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-10-25 02:44:04--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.1s    

2026-10-25 02:44:04 (10.2 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [2]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [3]:
print("char length: ",len(text))

char length:  1115394


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [6]:
chars = sorted(list(set(text))) # sorted list of all the char that occures
vocab_size = len(chars)  # size of vocabulary
print(''.join(chars))
print(vocab_size)

# 0 is '\n' and 1 is ''


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [7]:
# tokenization of input text (vocab to numbers)
# char level lang model
stoi = {ch:i for i, ch in enumerate(chars)}
itos = {i:ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of int, output a string

print(encode("hi there"))
print(decode(encode("hi there")))

# others:  openAI-tiktokens, google-Sentencebit

[46, 47, 1, 58, 46, 43, 56, 43]
hi there


In [8]:
# Encoding entire dataset

import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [9]:
# data split (tr: 90%, test:10%)
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [10]:
# how we train a transformer:
    # To compute heavy to do it all at once, so we sample random chunks and train iterativily on them
    # max len of chunks ( block_size )

In [11]:
# how transformers work? unlike RNNs and CNNs, transformers train the block_size in parallel not iterative
# So, a block size of 8 means, training 8 threads starting with a pair (1 prev predicts next) then (2 prev predicts
# next) ... total 8

block_size = 8
train_data[:block_size+1] # why +1, see above

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [13]:
# This is done so that transformers learns connections from as little as 1 input to block_size inputs
# So that sampling can be done from the begining without any reference

# this is the Time dimension: what comes next (pridicting future char)
x = train_data[:block_size] #(0, 7)
y = train_data[1:block_size + 1] #(1, 8)
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target is: {target}")

when input is tensor([18]) the target is: 47
when input is tensor([18, 47]) the target is: 56
when input is tensor([18, 47, 56]) the target is: 57
when input is tensor([18, 47, 56, 57]) the target is: 58
when input is tensor([18, 47, 56, 57, 58]) the target is: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target is: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target is: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target is: 58


In [14]:
# next dimension Batch dimension:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
  # genereate a small batch
  data = train_data if split == 'train' else val_data
  ix = torch.randint(len(data) - block_size, (batch_size,))  # where the batch starts
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])
  return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

for b in range(batch_size): # batch dim
  for t in range(block_size): # time dim
    context = xb[b, :t+1] # input to tranformer
    target = yb[b, t] # what to expect out of transformer
    print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [44, 53, 56,

In [15]:
# implementing a bigram language model
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigamLanaguageModel(nn.Module):  # BigamLanaguageModel is a subclass of nn.Module

  def __init__(self, vocab_size):
    super().__init__()
    # each token directly reads off the logits for the next token from the table lookup
    self.token_embedding_table = nn.Embedding(vocab_size, vocab_size) # creates a wrapper aroung tensor of given size

  def forward(self, idx, targets=None):  # method over-ridding ( defining how forward pass will happen)

    # idx ad targets are both (B, T) tensor of intergets
    logits = self.token_embedding_table(idx) # (B, T, C) (batch = batch_size, time = block_size, channels = vocab_size)

    if targets == None:
      loss = None
    else:

      # how to pass logits to F.cross_entropy? we have (B, T, C), F.cross_entropy expects C to be 2nd dim
      B, T, C = logits.shape
      logits = logits.view(B*T, C) # (4, 8, 65) -> (32, 65)
      targets = targets.view(B*T) # (4, 8) -> (32)
      loss = F.cross_entropy(logits, targets)

    return logits, loss

  def generate(self, idx, max_new_tokens):
    # idx is (B, T) array of indices in the current context
    for _ in range(max_new_tokens):
      # get the predictions
      logits, loss = self(idx) # perfroms __call__ on model
      # focus only on last time step
      logits = logits[ :, -1 ,: ] # becomes (B, C) : to keep it bigram (only look at just previous not the whole history)
      # apply softmax to get probs
      probs = F.softmax(logits, dim=1) # (B, C)
      # sample from the distribution
      idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
      # append sampled idx to the running sequence
      idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
    return idx


m = BigamLanaguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1,1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [16]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)  # what we used till now = SGT Schotastic gradient descent, now AdamW

In [ ]:
batch_size = 32
for steps in range(10000):

  # sample a batch of date
  xb, yb = get_batch('train') # (B, T, C) = (32, 8, 65)

  # evaluate the loss
  logits, loss = m(xb, yb) # forward pass
  optimizer.zero_grad(set_to_none=True) # gards to None
  loss.backward() # backward pass
  optimizer.step() # update the grads

print(loss.item())

2.382369041442871


In [ ]:
print(decode(m.generate(idx = torch.zeros((1,1), dtype=torch.long), max_new_tokens=500)[0].tolist()))


lso br. ave aviasurf my, yxMPZI ivee iuedrd whar ksth y h bora s be hese, woweee; the! KI 'de, ulseecherd d o blllando;LUCEO, oraingofof win!
RIfans picspeserer hee tha,
TOFonk? me ain ckntoty ded. bo'llll st ta d:
ELIS me hurf lal y, ma dus pe athouo
BEY:! Indy; by s afreanoo adicererupa anse tecorro llaus a!
OLeneerithesinthengove fal amas trr
TI ar I t, mes, n IUSt my w, fredeeyove
THek' merer, dd
We ntem lud engitheso; cer ize helorowaginte the?
Thak orblyoruldvicee chot, p,
Bealivolde Th li


In [18]:
# maths of self-attention

# B = batch, T= time, C= channel info
# now we would like to couple them, but as we will predict the next token, coupling with future will eventually give away the answer, so we should only couple with past timestamps. So at T = 5, we have context of 5 to predict the 6h char. But we also have the T [1, 4], already.

# So the easiest way to couple them is to average up all the past info till current and use the generated feature vector to predict future (next char)

# although a avg is very simple, it's alright for now. There are other ways to it - for further study

torch.manual_seed(1337)
B,T,C = 4, 8, 2
x = torch.randn(B, T, C)
x.shape

torch.Size([4, 8, 2])

In [19]:
# we want x[b,t] = mean_{i<=t} x[b, i]
xbow = torch.zeros((B, T, C)) # bag of words
for b in range(B):
  for t in range(T):
    xprev = x[b,: t+1] # (t, C)
    xbow[b,t] = torch.mean(xprev, dim=0) # mean at t(dim =0) so resulting matrix - (1, C)

# Very inefficient
# using matrix mult is much more efficient

In [20]:
# trick
torch.tril(torch.ones(3, 3))

tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])

In [21]:
# due to the nature of matrix mult, we can use an lower triangular matrix and mult it with x to get the sum of x's from past to current row. then we just have to divide each row by the scalar row(row_number) to get the average.

# earlier
# for b in range(B):
#   for t in range(T):
#     xprev = x[b,: t+1] # (t, C)
#     xbow[b,t] = torch.mean(xprev, dim=0)

# version 2
wei = torch.tril(torch.ones((T, T)))
wei = wei / wei.sum(dim=-1, keepdim=True) # for mean
xbow2 = wei @ x # (T, T) @ (B, T, C) ----> (B, T, C)
print("Max absolute difference:", torch.max(torch.abs(xbow - xbow2)).item()) # identical



Max absolute difference: 3.236345946788788e-08


In [22]:
# version 3: use softmax
# masking using tril on wei, where tril == 0, wei = -inf (so wei is reverse lower triangular matrix with '-inf' instead of 1
# as e^0 = 1 and e^'-inf' = 0
# as all the now '-inf' valuse of wei is same (0) doing softmax, it results in averaging of the row
tril = torch.tril(torch.ones((T, T)))
wei = torch.zeros(T, T)
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x # (T, T) @ (B, T, C) ----> (B, T, C)  # simple avg
print("Max absolute difference:", torch.max(torch.abs(xbow - xbow3)).item())

Max absolute difference: 3.236345946788788e-08


In [23]:
# Why use the softmax approach instead of normal avg

# Because of what it represents in training
# Afinity - in the begining all 0 wei(weights) means on init there is no afininty bw any of the tokens
# But as the model will learn , these weight will change based on how interesting they find each other

# And then we use masking to make sure the model only interacts with the past and present info

In [26]:
wei[0]

tensor([[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00],
        [5.2316e-01, 4.7684e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00],
        [6.8504e-01, 2.0609e-01, 1.0887e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00],
        [3.0207e-02, 1.9260e-02, 6.9410e-01, 2.5643e-01, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00],
        [5.0116e-02, 1.2298e-01, 5.3247e-01, 2.5875e-01, 3.5689e-02, 0.0000e+00,
         0.0000e+00, 0.0000e+00],
        [5.1325e-01, 4.2650e-01, 5.1536e-02, 7.0999e-03, 1.3650e-03, 2.4790e-04,
         0.0000e+00, 0.0000e+00],
        [1.5800e-01, 1.9924e-01, 1.0625e-01, 1.3482e-01, 1.3614e-01, 1.5542e-01,
         1.1013e-01, 0.0000e+00],
        [1.7153e-01, 3.7644e-02, 9.0361e-02, 6.1836e-02, 1.8533e-01, 6.3256e-02,
         3.3406e-01, 5.5985e-02]], grad_fn=<SelectBackward0>)

In [27]:
# version 4: self attention


B, T, C = 4, 8, 32
x = torch.randn(B, T, C)

# a single Head perform self-attention
head_size = 16  # hyper-param

key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
k = key(x) # (B, T, head_size)
q = query(x) # (B, T, head_size)

value = nn.Linear(C, head_size, bias=False)

wei = q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) ---> (B, T, T)

tril = torch.tril(torch.ones( (T, T)))
# wei = torch.zeros(T, T)  # now we don't jsut want this to be zeros, we want them to be data dependent (function of keys and queries)
# What data? see below

wei = wei.masked_fill(tril == 0, float('-inf')) # do not agregate future info
wei = F.softmax(wei, dim=-1) # how much info to agregate (based on qyery and key alignment)

v = value(x) # (B, T, head_size) # what c will communicate
out = wei @ v # (B, T, T) @ (B, T, head_size) ---> (B, T, head_size)
# out = wei @ x  # x is private to the token
# Why use Value instead of x? see below

out.shape

# What data?

# in self attention - we want eact position to store 2 things - query and key
# query - what am i looking for
# key - what do i have ( already know )

# then wei will just be dot product bw queries and keys

# so as we learn - the keys and queries vector will align more to tokens that are releted then other tokens in the sequence

# Why use Value instead of x?

# Each head can specialize. With multiple heads (when we build it), each has its own value layer, so head 1 might pass along syntactic info and head 2 positional or semantic info. If they all used x, every head would be averaging the same vectors and differ only in the weights, which is much less expressive.

# x becomes private to the token (it's identity), so for a single head - x tells - ( k - here is what i have, q - here is what i am looking for, v - hear is what i'll communicate with you)

torch.Size([4, 8, 16])

Notes:

1. What is attention?
It is a communication mechanism. Can be seen as nodes in a directed graph, where each node looks at the nodes pointing to it and aggregates their information as a weighted sum. The weights are data-dependent: they're computed on the fly from q·k, not fixed parameters.

In our decoder: node t has edges from nodes 0..t (itself + the past), which is what the tril mask enforces. So token 0 sees only itself, token T-1 sees everything.

2. also, there is no notion of space, Attention simply acts over a set of vectors. This is why we need to positionally encode tokens. so that they know wehre the are and based on their info.

3. each example across batch dimention is porcessed completely independently and never "talk" to each other

4. In an "encoder" attention block just delete the single line that does masking with tril, allowing all tokens to communicate. This block here is called a "decoder" attention block because it has triangular masking, and is usually used in autoregressive settings, like language modeling.

5. "self-attention" just means that the keys and values are produced from the same source as queries (in our case - x). In "cross-attention", the queries still get produced from x, but the keys and values come from some other, external source (e.g. an encoder module)

6. "Scaled" attention additional divides wei by 1/sqrt(head_size). This makes it so when input Q,K are unit variance, wei will be unit variance too and Softmax will stay diffuse and not saturate too much. Illustration below

In [42]:
# Scaled attention

k = torch.randn(B, T, head_size)
q = torch.randn(B, T, head_size)
# wei = q @ k.transpose(-2, -1) # variance is aprox head_size
wei = q @ k.transpose(-2, -1) * head_size**-0.5 # varience is preserved at aprox 1

# why?
# because wei is going to be feeded into softmax, so it's important that wie be somewhat diffused at init as if not, the resulting softmax we be approx a one-hot vector (i.e. one entry close to 1, rest approx 0)

In [43]:
k.var()

tensor(0.8828)

In [44]:
q.var()

tensor(1.0159)

In [45]:
wei.var()

tensor(1.0327)

In [37]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim=-1)

tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])

2 Major Ideas for optimizing deep neural nets:

1. Skip/Residual Connections:
  https://medium.com/data-science/residual-blocks-building-blocks-of-resnet-fd90ca15d6ec

So, if I am understanding it correctly, what we are trying to say is it's kind of like not start with zero or not predict the whole thing, but consider what the given input is and try to predict what should go on top of that?

Exactly. Instead of building the output from scratch, each block starts from what it was given and only predicts the change: “keep x, and here’s what to add to it.”

One refinement that fits the transformer especially well: think of x as a running stream of information flowing through the whole network. Each block reads from the stream, computes something (attention gathers info from other tokens, the feed-forward layer processes it), and writes its contribution back by adding it. Nothing gets overwritten, so later blocks still have access to everything earlier blocks put there.

And if a block has nothing useful to add, it can just output roughly zero and the stream passes through untouched. That’s the “default is identity” property that makes deep networks easy to train.

2. Layernorm: see below

In [50]:
class LayerNorm1d:

    def __init__ (self, dim, eps= 1e-5, momentum=0.1):
        self.eps = eps
        # parameters (training with backprop)
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)

    def __call__ (self, x):
        xmean = x.mean(1, keepdim=True)   # layer mean
        xvar = x.var(1, keepdim=True, unbiased=True)     # layer std

        xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit vector
        self.out = self.gamma * xhat + self.beta
        return self.out

    def parameters(self):
        return [self.gamma, self.beta]

torch.manual_seed(1337)
module = LayerNorm1d(100)
x = torch.randn(32, 100)
x = module(x)
x.shape

torch.Size([32, 100])

In [51]:
x[:,0].mean(), x[:,0].std()  # batch_norm normalizes column

(tensor(0.1469), tensor(0.8803))

In [52]:
x[0,:].mean(), x[0,:].std() # Layer norm normalizes rows

(tensor(-9.5367e-09), tensor(1.0000))

How to prevent overfitting:

1. Dropout: https://www.cs.toronto.edu/~rsalakhu/papers/srivastava14a.pdf

- During training, each forward/backward pass randomly sets a fraction p of activations to 0. A different random subset every pass.
- Effect: you're training a huge ensemble of thinned subnetworks that share weights. At test time you use the full network, which acts like an average of that ensemble.
- Why it helps: neurons can't rely on specific other neurons always being there (prevents "co-adaptation"), so features must be useful on their own. It's a regularizer: reduces overfitting.
- Only active in training. model.train() turns it on, model.eval() turns it off.
- PyTorch uses "inverted dropout": surviving activations are scaled by 1/(1-p) during training so the expected value stays the same, and nothing needs rescaling at inference.

Where it goes in our GPT (not before every layer):
- on wei after softmax (randomly blocks some tokens from communicating)
- after the projection at the end of multi-head attention
- after the last Linear in the feed-forward block

What we do not implement from Attention is all you need paper:

1. No Encoder block for transformer
2. No cross-attention in the decoder block, just self-attertion

Why decoder only:
- We are just generating text. i.e. just next char prediction.

When encoder+decoder transformers are used:
- if the generation is based on some input. In the paper, they are translating French to English, so generation is based on the given text.
 - For us, it's just similar text generation based on training
- So, the key and values are coming from encoder and the queries are solely based on decoder